# Evaluation


In [1]:
import sys
from pathlib import Path

# Find the project root by walking up until src/olist is visible, so the
# notebook runs from any working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import pandas as pd
from olist import config as cfg, modeling as ml

pd.set_option("display.max_columns", 200); pd.set_option("display.width", 250)

PROJECTS = ["01_late_delivery_classification", "02_delivery_time_regression",
            "03_freight_cost_prediction", "04_delivery_outcome_multiclass",
            "05_order_service_segmentation"]

board = ml.scorecard(PROJECTS)
missing = board.loc[board["status"] != "trained", "project"].tolist()
if missing:
    print("NOT TRAINED YET (run the notebooks in notebooks/modeling/ first):")
    for m in missing: print("  -", m)
    print()
board

,project,status,model,target,n_features,test_accuracy,test_balanced_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,test_brier,baseline_rules_accuracy,baseline_rules_balanced_accuracy,baseline_rules_precision,baseline_rules_recall,baseline_rules_f1,test_mae,test_rmse,test_r2,test_mape_pct,test_median_abs_err,baseline_incumbent_promise_mae,baseline_incumbent_promise_rmse,baseline_incumbent_promise_r2,baseline_incumbent_promise_mape_pct,baseline_incumbent_promise_median_abs_err,baseline_rate_card_mae,baseline_rate_card_rmse,baseline_rate_card_r2,baseline_rate_card_mape_pct,baseline_rate_card_median_abs_err,test_macro_f1,test_weighted_f1,test_macro_precision,test_macro_recall,test_cohen_kappa,test_macro_roc_auc,baseline_majority_accuracy,baseline_majority_balanced_accuracy,baseline_majority_macro_f1,baseline_majority_weighted_f1,baseline_majority_macro_precision,baseline_majority_macro_recall,baseline_majority_cohen_kappa,baseline_majority_macro_roc_auc
0,01_late_delivery_classification,trained,logreg,late_delivery_flag,39.0,0.9190,0.5119,0.223,0.0331,0.0577,0.524,0.0908,0.0758,0.2196,0.5611,0.0848,0.9626,0.1559,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02_delivery_time_regression,trained,hist_gbm,delivery_days,39.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0914,5.6332,-0.0999,61.9950,3.0637,9.7225,12.0912,-4.0671,153.0381,8.2244,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03_freight_cost_prediction,trained,ridge,order_freight_total,36.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.6635,2.0330,0.9945,2.3086,0.3044,NaN,NaN,NaN,NaN,NaN,16.3378,25.3062,0.1457,69.9557,12.2331,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04_delivery_outcome_multiclass,trained,random_forest,outcome_band,39.0,0.5323,0.3113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.2851,0.5347,0.2988,0.3113,0.2982,0.6131,0.6993,0.2,0.1646,0.5755,0.1399,0.2,0.0,0.5
4,05_order_service_segmentation,trained,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Every project next to its own baseline. A model without its baseline is a claim
# without evidence, so this table refuses to show one without the other.
rows = []
for project in PROJECTS:
    path = cfg.MODELS_DIR / project / "model_metadata.json"
    if not path.exists():
        continue
    meta = json.loads(path.read_text(encoding="utf-8"))
    metrics = meta.get("metrics_test") or {}
    baseline = meta.get("metrics_baseline_test") or {}
    headline = next((k for k in ("pr_auc", "macro_f1", "mae") if k in metrics), None)
    base_key = next((k for k in baseline if headline and k.endswith(headline)), None)
    rows.append({
        "project": project,
        "model": meta.get("model"),
        "target": meta.get("target"),
        "features": meta.get("n_features"),
        "headline metric": headline,
        "model score": round(metrics[headline], 4) if headline else None,
        "baseline score": round(baseline[base_key], 4) if base_key else None,
        "policy": (meta.get("decision_policy") or {}).get("type", "n/a"),
    })
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

                        project         model              target  features headline metric  model score  baseline score                  policy
01_late_delivery_classification        logreg  late_delivery_flag      39.0          pr_auc       0.0908             NaN fixed_expedite_capacity
    02_delivery_time_regression      hist_gbm       delivery_days      39.0             mae       4.0914          9.7225        quantile_promise
     03_freight_cost_prediction         ridge order_freight_total      36.0             mae       0.6635         16.3378         cost_estimation
 04_delivery_outcome_multiclass random_forest        outcome_band      39.0        macro_f1       0.2851          0.1646    service_tier_routing
  05_order_service_segmentation          None                None       NaN            None          NaN             NaN                     n/a


In [3]:
# Reproducibility audit: can a stranger rebuild every number in this repo?
checks = []
for project in PROJECTS:
    directory = cfg.MODELS_DIR / project
    meta_path = directory / "model_metadata.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    checks.append({
        "project": project,
        "model file": (directory / "best_model.joblib").exists(),
        "metadata": meta_path.exists(),
        "feature list stored": bool(meta.get("features")),
        "seed stored": meta.get("random_state") is not None,
        "snapshot date stored": bool(meta.get("snapshot_date")),
        "split documented": bool(meta.get("split") or meta.get("validation")),
        "decision policy stored": bool(meta.get("decision_policy") or meta.get("segment_names")),
    })
repro = pd.DataFrame(checks).set_index("project")
print(repro.to_string())
print(f"\nall boxes ticked: {bool(repro.all().all())}")

                                 model file  metadata  feature list stored  seed stored  snapshot date stored  split documented  decision policy stored
project                                                                                                                                                
01_late_delivery_classification        True      True                 True         True                  True              True                    True
02_delivery_time_regression            True      True                 True         True                  True              True                    True
03_freight_cost_prediction             True      True                 True         True                  True              True                    True
04_delivery_outcome_multiclass         True      True                 True         True                  True              True                    True
05_order_service_segmentation          True      True                 True         True 